In [3]:
# imports
import numpy as np
import pandas as pd

#creating document corpus
data = {
    "doc_id": [1, 2, 3],
    "text": [
        "Machine learning is a subset of artificial intelligence.",
        "RAG stands for Retrieval Augmented Generation.",
        "Large Language Models are trained on large datasets."
    ]
}
corpus = pd.DataFrame(data)

# text preprocessing
df = corpus.copy()
print(df)
print("-------"*10)
df["text_clean"] = df["text"].str.lower()
print(df["text_clean"])

# tokenization
vocab = set()

for sentence in df["text_clean"]:
  for word in sentence.split():
    vocab.add(word)
vocab = sorted(list(vocab))
print("-------"*10)
print(vocab)

# text -> vector (embedding)
def text_to_vector(text,vocab):
  vector = np.zeros(len(vocab))
  for word in text.split():
    if word in vocab:
      vector[vocab.index(word)]+=1
  return vector

#generationg document embeddings
doc_embedding = np.array(
[text_to_vector(text,vocab) for text in df["text_clean"]]

)
print("-------"*10)
print(doc_embedding)
print("-------"*10)
print(doc_embedding.shape)

query = "what is Large language models?"
query_clean = query.lower()
query_vector = text_to_vector(query_clean,vocab)

# retriever using cosine

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0:
        return 0.0
    return np.dot(vec1, vec2) / denom

#computing similarity in all documents
similarities = []

for doc_vec in doc_embedding:
    sim = cosine_similarity(query_vector, doc_vec)
    similarities.append(sim)
print("-----"*10)
df["similarity"] = similarities
print(df[["text", "similarity"]])

# retrieve top-k
top_k = df.sort_values(by="similarity",ascending=False).head(1)
retrived_content = top_k["text"].values[0]
print("-----"*10)
print(f"Retrived content:{retrived_content}")

#Generator
def generate_answer(query, context):
    return f"Answer based on retrieved context:\n{context}"
answer = generate_answer(query, retrived_content)
print(answer)

   doc_id                                               text
0       1  Machine learning is a subset of artificial int...
1       2     RAG stands for Retrieval Augmented Generation.
2       3  Large Language Models are trained on large dat...
----------------------------------------------------------------------
0    machine learning is a subset of artificial int...
1       rag stands for retrieval augmented generation.
2    large language models are trained on large dat...
Name: text_clean, dtype: object
----------------------------------------------------------------------
['a', 'are', 'artificial', 'augmented', 'datasets.', 'for', 'generation.', 'intelligence.', 'is', 'language', 'large', 'learning', 'machine', 'models', 'of', 'on', 'rag', 'retrieval', 'stands', 'subset', 'trained']
----------------------------------------------------------------------
[[1. 0. 1. 0. 0. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0.]